# Rung 31 — attention probe

**The run generator.** Config is inline; the engine is `_tools/attention_probe.py` and is
imported, never launched by hand.

🔴 **One arm per execution, on purpose.** Four 8B models do not fit in one process and
`del model` + `empty_cache()` does not free them — PEFT wraps the base and nothing is
collected until the GC runs, so arm 2 loads its 16 GB beside arm 1's and OOMs a 32 GB card
(measured 2026-08-08). Process exit is the only guaranteed teardown, so this notebook is
executed **once per arm** and a crash in arm 3 cannot destroy arms 1 and 2.

Headless, detached, one arm at a time, then the merge:

```bash
cd /workspace/repo_leo/experiments/31-attention-probe
for ARM in base rung02 rung06 a2; do
  papermill 31_attention_probe.ipynb /workspace/tmp/leo_31_${ARM}.ipynb \
    -p ARM $ARM -p MERGE False --log-output
done
papermill 31_attention_probe.ipynb /workspace/tmp/leo_31_merge.ipynb -p MERGE True --log-output
```

⚠️ `pgrep -f "papermill 31_attention_probe"` matches its own `bash -c` wrapper — use
`[p]apermill` or a wait loop built on it never exits.

In [ ]:
# papermill parameters
ARM = "base"          # base | rung02 | rung06 | a2
N_QUESTIONS = 12      # paired: the SAME rows for every arm
MAX_PIXELS = 512 * 512
SEED = 42
MERGE = False         # True = skip the run, just combine the per-arm JSONs

In [ ]:
import json
import sys
from pathlib import Path

EXP = Path.cwd()
sys.path.insert(0, str(EXP / "_tools"))

from attention_probe import Config, main, merge

cfg = Config(n_questions=N_QUESTIONS, max_pixels=MAX_PIXELS, seed=SEED)
print(f"arm={ARM} merge={MERGE} max_pixels={cfg.max_pixels} out={cfg.out}")

In [ ]:
result = merge(cfg) if MERGE else main(cfg, ARM)
print(json.dumps(result.get("agg", result), indent=2)[:2000])